In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

In [3]:
print(movies.head())
print(ratings.head())

print(movies.shape)
print(ratings.shape)

print(movies.isnull().sum())
print(ratings.isnull().sum())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
(9742, 3)
(100836, 4)
movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [4]:
data = pd.merge(ratings, movies, on="movieId")
print(data.head())

# Count how many ratings each movie has
movie_rating_counts = data.groupby("title")["rating"].count()

# Keep only movies with at least 50 ratings
popular_movies = movie_rating_counts[movie_rating_counts >= 50].index

# Filter dataset
filtered_data = data[data["title"].isin(popular_movies)]

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


In [5]:
user_movie_matrix = filtered_data.pivot_table(index="userId", columns="title", values="rating")
print(user_movie_matrix.head())

title   10 Things I Hate About You (1999)  12 Angry Men (1957)  \
userId                                                           
1                                     NaN                  NaN   
2                                     NaN                  NaN   
3                                     NaN                  NaN   
4                                     NaN                  5.0   
5                                     NaN                  NaN   

title   2001: A Space Odyssey (1968)  28 Days Later (2002)  300 (2007)  \
userId                                                                   
1                                NaN                   NaN         NaN   
2                                NaN                   NaN         NaN   
3                                NaN                   NaN         NaN   
4                                NaN                   NaN         NaN   
5                                NaN                   NaN         NaN   

title   40-Year-Ol

In [6]:
print("Users:", filtered_data["userId"].nunique())
print("Movies:", filtered_data["movieId"].nunique())
print("Ratings:", len(filtered_data))
print("Matrix shape:", user_movie_matrix.shape)

Users: 606
Movies: 451
Ratings: 41362
Matrix shape: (606, 450)


In [7]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [8]:
movie_matrix = user_movie_matrix_filled.T
movie_similarity = cosine_similarity(movie_matrix)
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_matrix.index,
    columns=movie_matrix.index
)

print(movie_similarity_df.head())

title                              10 Things I Hate About You (1999)  \
title                                                                  
10 Things I Hate About You (1999)                           1.000000   
12 Angry Men (1957)                                         0.095586   
2001: A Space Odyssey (1968)                                0.191393   
28 Days Later (2002)                                        0.170720   
300 (2007)                                                  0.351032   

title                              12 Angry Men (1957)  \
title                                                    
10 Things I Hate About You (1999)             0.095586   
12 Angry Men (1957)                           1.000000   
2001: A Space Odyssey (1968)                  0.291756   
28 Days Later (2002)                          0.217353   
300 (2007)                                    0.261876   

title                              2001: A Space Odyssey (1968)  \
title                

In [9]:
def recommend_movies(movie_title, similarity_df, n=5):
    if movie_title not in similarity_df.columns:
        matches = [title for title in similarity_df.columns if movie_title.lower() in title.lower()]
        return f"Movie not found. Possible matches: {matches[:5]}"

    similar_scores = similarity_df[movie_title].sort_values(ascending=False).iloc[1:n+1]
    return pd.DataFrame({
        "Recommended Movie": similar_scores.index,
        "Similarity Score": similar_scores.values
    })

In [10]:
print("Recommendations for Toy Story (1995):")
print(recommend_movies("Toy Story (1995)", movie_similarity_df, 5))

print("\nRecommendations for Jurassic Park (1993):")
print(recommend_movies("Jurassic Park (1993)", movie_similarity_df, 5))

Recommendations for Toy Story (1995):
                           Recommended Movie  Similarity Score
0                         Toy Story 2 (1999)          0.572601
1                       Jurassic Park (1993)          0.565637
2       Independence Day (a.k.a. ID4) (1996)          0.564262
3  Star Wars: Episode IV - A New Hope (1977)          0.557388
4                        Forrest Gump (1994)          0.547096

Recommendations for Jurassic Park (1993):
                   Recommended Movie  Similarity Score
0  Terminator 2: Judgment Day (1991)          0.719983
1                Forrest Gump (1994)          0.688259
2                  Braveheart (1995)          0.669454
3               Fugitive, The (1993)          0.668804
4                       Speed (1994)          0.661732


In [11]:
print("Number of users:", filtered_data["userId"].nunique())
print("Number of movies:", filtered_data["movieId"].nunique())
print("Number of ratings:", len(filtered_data))

top_movies = data.groupby("title")["rating"].count().sort_values(ascending=False).head(10)
print(top_movies)

Number of users: 606
Number of movies: 451
Number of ratings: 41362
title
Forrest Gump (1994)                          329
Shawshank Redemption, The (1994)             317
Pulp Fiction (1994)                          307
Silence of the Lambs, The (1991)             279
Matrix, The (1999)                           278
Star Wars: Episode IV - A New Hope (1977)    251
Jurassic Park (1993)                         238
Braveheart (1995)                            237
Terminator 2: Judgment Day (1991)            224
Schindler's List (1993)                      220
Name: rating, dtype: int64


In [12]:
print(recommend_movies("Forrest Gump (1994)", movie_similarity_df, 5))
print(recommend_movies("Matrix, The (1999)", movie_similarity_df, 5))
print(recommend_movies("Pulp Fiction (1994)", movie_similarity_df, 5))

                  Recommended Movie  Similarity Score
0  Shawshank Redemption, The (1994)          0.712993
1              Jurassic Park (1993)          0.688259
2               Pulp Fiction (1994)          0.685544
3                 Braveheart (1995)          0.643090
4  Silence of the Lambs, The (1991)          0.639463
                                   Recommended Movie  Similarity Score
0                                  Fight Club (1999)          0.713937
1  Star Wars: Episode V - The Empire Strikes Back...          0.700935
2                         Saving Private Ryan (1998)          0.679615
3          Star Wars: Episode IV - A New Hope (1977)          0.663447
4  Star Wars: Episode VI - Return of the Jedi (1983)          0.660984
                  Recommended Movie  Similarity Score
0  Silence of the Lambs, The (1991)          0.709382
1  Shawshank Redemption, The (1994)          0.702366
2       Seven (a.k.a. Se7en) (1995)          0.697654
3               Forrest Gump (1994

In [13]:
total_cells = user_movie_matrix.shape[0] * user_movie_matrix.shape[1]
rated_cells = user_movie_matrix.count().sum()
sparsity = 1 - (rated_cells / total_cells)

print("Sparsity:", sparsity)

Sparsity: 0.8483314998166483


In [14]:
# Dataset summary
num_users = filtered_data["userId"].nunique()
num_movies = filtered_data["movieId"].nunique()
num_ratings = len(filtered_data)
matrix_rows, matrix_cols = user_movie_matrix.shape
sparsity = 1 - (user_movie_matrix.count().sum() / (matrix_rows * matrix_cols))

summary_df = pd.DataFrame(
    {
        "Value": [num_users, num_movies, num_ratings, matrix_rows, matrix_cols, sparsity]
    },
    index=["Users", "Movies", "Ratings", "Matrix Rows", "Matrix Columns", "Sparsity"]
)

print("Dataset summary:\n")
print(summary_df)

# Popularity baseline
movie_popularity = filtered_data.groupby("title")["rating"].count().sort_values(ascending=False)
print("\nTop 5 most-rated movies (popularity baseline):")
print(movie_popularity.head(5))

def popularity_recommend(seed_title, popularity_series, n=5):
    return [title for title in popularity_series.index if title != seed_title][:n]


Dataset summary:

                       Value
Users             606.000000
Movies            451.000000
Ratings         41362.000000
Matrix Rows       606.000000
Matrix Columns    450.000000
Sparsity            0.848331

Top 5 most-rated movies (popularity baseline):
title
Forrest Gump (1994)                 329
Shawshank Redemption, The (1994)    317
Pulp Fiction (1994)                 307
Silence of the Lambs, The (1991)    279
Matrix, The (1999)                  278
Name: rating, dtype: int64


In [15]:
seed_movies = ["Toy Story (1995)", "Forrest Gump (1994)", "Matrix, The (1999)", "Pulp Fiction (1994)"]

for seed in seed_movies:
    print("---")
    print(f"Seed movie: {seed}")
    print("Popularity baseline:")
    print(popularity_recommend(seed, movie_popularity, 5))
    print("Similarity-based recommendations:")
    try:
        recs = recommend_movies(seed, movie_similarity_df, 5)
        print(list(recs["Recommended Movie"]))
    except Exception as exc:
        print(f"Recommendation error: {exc}")


---
Seed movie: Toy Story (1995)
Popularity baseline:
['Forrest Gump (1994)', 'Shawshank Redemption, The (1994)', 'Pulp Fiction (1994)', 'Silence of the Lambs, The (1991)', 'Matrix, The (1999)']
Similarity-based recommendations:
['Toy Story 2 (1999)', 'Jurassic Park (1993)', 'Independence Day (a.k.a. ID4) (1996)', 'Star Wars: Episode IV - A New Hope (1977)', 'Forrest Gump (1994)']
---
Seed movie: Forrest Gump (1994)
Popularity baseline:
['Shawshank Redemption, The (1994)', 'Pulp Fiction (1994)', 'Silence of the Lambs, The (1991)', 'Matrix, The (1999)', 'Star Wars: Episode IV - A New Hope (1977)']
Similarity-based recommendations:
['Shawshank Redemption, The (1994)', 'Jurassic Park (1993)', 'Pulp Fiction (1994)', 'Braveheart (1995)', 'Silence of the Lambs, The (1991)']
---
Seed movie: Matrix, The (1999)
Popularity baseline:
['Forrest Gump (1994)', 'Shawshank Redemption, The (1994)', 'Pulp Fiction (1994)', 'Silence of the Lambs, The (1991)', 'Star Wars: Episode IV - A New Hope (1977)']
S